In [ ]:
# 1. Data is Balanced using 5 Balancing Techniques and Visualized in Lolipop Chart

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.under_sampling import TomekLinks

# 1. Load & Prepare Data
df = pd.read_csv('heart.csv')
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=369)

# 2. Define Techniques
samplers = {
    "Original": None,
    "SMOTE": SMOTE(random_state=369),
    "ADASYN": ADASYN(random_state=369),
    "SMOTETomek": SMOTETomek(random_state=369),
    "Tomek Links": TomekLinks(),
    "SMOTEENN": SMOTEENN(random_state=369)
}

# 3. Process Data
stats = []
for name, sampler in samplers.items():
    if sampler is None:
        counts = y_train.value_counts()
    else:
        try:
            X_res, y_res = sampler.fit_resample(X_train, y_train)
            counts = y_res.value_counts()
        except:
            continue
    stats.append({"Technique": name, "Class": 0, "Count": counts.get(0, 0)}) # Normal
    stats.append({"Technique": name, "Class": 1, "Count": counts.get(1, 0)}) # Disease

df_stats = pd.DataFrame(stats)

# ---------------------------------------------------------
# 4. PLOTTING
# ---------------------------------------------------------
plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")

# Define Positions
techniques = df_stats['Technique'].unique()
y_range = range(len(techniques))
offsets = [-0.15, 0.15]

# Colors: Navy Blue & Crimson Red
colors = ["#1ABC9C", "#C0392B"]

for i, tech in enumerate(techniques):
    row_norm = df_stats[(df_stats['Technique']==tech) & (df_stats['Class']==0)]
    row_dis = df_stats[(df_stats['Technique']==tech) & (df_stats['Class']==1)]

    val_norm = row_norm['Count'].values[0]
    val_dis = row_dis['Count'].values[0]

    # --- Draw Stem 1: Normal ---
    plt.hlines(y=i + offsets[0], xmin=0, xmax=val_norm, color=colors[0], alpha=0.9, linewidth=4.5)
    plt.plot(val_norm, i + offsets[0], "o", markersize=15, color=colors[0])
    plt.text(val_norm + 18, i + offsets[0], str(val_norm), va='center', fontsize=11, fontweight='bold', color=colors[0])

    # --- Draw Stem 2: Disease ---
    plt.hlines(y=i + offsets[1], xmin=0, xmax=val_dis, color=colors[1], alpha=0.9, linewidth=4.5)
    plt.plot(val_dis, i + offsets[1], "o", markersize=15, color=colors[1])
    plt.text(val_dis + 18, i + offsets[1], str(val_dis), va='center', fontsize=11, fontweight='bold', color=colors[1])

# Styling
plt.yticks(y_range, techniques, fontsize=13, fontweight='bold')
plt.xlabel("Number of Samples", fontsize=13, labelpad=10, fontweight='bold')

# Clean Spines
sns.despine(left=True, bottom=True)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.grid(axis='y', alpha=0)

# --- LEGEND FIXED HERE ---
# Manually creating handles for the legend to place it exactly where we want
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Normal', markerfacecolor=colors[0], markersize=12),
    Line2D([0], [0], marker='o', color='w', label='Heart Disease', markerfacecolor=colors[1], markersize=12)
]

plt.legend(handles=legend_elements, loc='upper right', frameon=True, fontsize=12, facecolor='white', framealpha=1, edgecolor='black')

plt.tight_layout()
plt.show()

In [ ]:
# 2. Applying 5 type balanced dataset on each 5 ML Models and getting their evalaution metrics

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

# Balancing Libraries
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.under_sampling import TomekLinks

# Load Data
df = pd.read_csv('heart.csv')

# Preprocessing
le = LabelEncoder()
categorical_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# Split (SEED 369)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=369)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define Models (ALL SEED 369)
models = {
    "Random Forest": RandomForestClassifier(random_state=369),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=369),
    "LightGBM": LGBMClassifier(random_state=369, verbose=-1),
    "MLP": MLPClassifier(max_iter=1000, random_state=369),
    "SVC": SVC(probability=True, random_state=369)
}

# Define Techniques
samplers = {
    "SMOTE": SMOTE(random_state=369),
    "SMOTETomek": SMOTETomek(random_state=369),
    "ADASYN": ADASYN(random_state=369),
    "Tomek Links": TomekLinks(),
    "SMOTEENN": SMOTEENN(random_state=369)
}

# Run Experiment
print(f"{'Model':<15} | {'Technique':<12} | {'Accuracy':<9} | {'Precision':<9} | {'Recall':<9} | {'F1 Score':<9}")
print("-" * 75)

for model_name, model in models.items():
    for tech_name, sampler in samplers.items():
        try:
            # A. Balance Data (Training Only)
            X_res, y_res = sampler.fit_resample(X_train_scaled, y_train)

            # B. Train Model (Reset model each time to be safe)
            # We must re-initialize the model to ensure it learns from scratch
            model.fit(X_res, y_res)

            # C. Predict
            y_pred = model.predict(X_test_scaled)

            # D. Metrics
            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred)
            rec = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)

            print(f"{model_name:<15} | {tech_name:<12} | {acc:.4f}    | {prec:.4f}    | {rec:.4f}    | {f1:.4f}")

        except Exception as e:
            print(f"{model_name:<15} | {tech_name:<12} | FAILED: {e}")

    print("-" * 75) # Separator between models

In [ ]:
# 3. Line Graph Showing the Impact of Balancing on ML Models

import matplotlib.pyplot as plt

# Data Setup
models = ['Random Forest', 'XGBoost', 'LightGBM', 'MLP', 'SVM', 'Proposed Ensemble']

# Data Structure: [Before, After]
data = {
    'Accuracy': {
        'Before': [87.59, 86.67, 87.22, 88.58, 86.13, 86.95],
        'After':  [90.22, 92.93, 92.93, 92.38, 92.38, 93.48],
        'Color': '#1f77b4',
        'Marker': 'o'
    },
    'Precision': {
        'Before': [88.61, 87.67, 88.22, 91.17, 88.28, 87.95],
        'After':  [91.43, 91.82, 91.07, 91.08, 91.08, 91.89],
        'Color': '#ff7f0e',
        'Marker': '^'
    },
    'Recall': {
        'Before': [86.59, 86.70, 86.22, 88.57, 87.13, 86.46],
        'After':  [89.13, 96.19, 97.14, 88.04, 88.04, 97.14],
        'Color': '#2ca02c',
        'Marker': 's'
    },
    'F1-Score': {
        'Before': [88.60, 88.67, 88.22, 89.85, 88.16, 88.45],
        'After':  [90.48, 93.95, 94.01, 89.52, 89.52, 94.44],
        'Color': '#d62728',
        'Marker': 'D'
    }
}

#  Plot Configuration
plt.figure(figsize=(14, 9), dpi=300)
plt.style.use('seaborn-v0_8-whitegrid')

#  Plotting Loop
for metric, values in data.items():
    # Plot "Before" (Thicker, Custom Dashing)
    line, = plt.plot(models, values['Before'],
             label=f'{metric} (Before)',
             color=values['Color'],
             linewidth=2.5,
             alpha=0.7,
             marker=values['Marker'],
             markerfacecolor='white',
             markersize=8,
             markeredgewidth=2)

    # Apply Custom Dash Pattern: (ink, space) -> (4, 4) gives wide spacing
    line.set_dashes([4, 4])

    # Plot "After" (Solid)
    plt.plot(models, values['After'],
             label=f'{metric} (After)',
             color=values['Color'],
             linestyle='-',
             linewidth=2.5,
             marker=values['Marker'],
             markersize=9)

    # Annotate ONLY the Proposed Ensemble for the "After" condition
    idx = 5
    val = values['After'][idx]

    # Slight vertical offset adjustment to prevent text overlapping
    offset = 12
    if metric == 'Accuracy': offset = -20
    if metric == 'Precision': offset = -20

    plt.annotate(f"{val}", (idx, val),
                 xytext=(0, offset), textcoords="offset points",
                 ha='center', fontsize=10, fontweight='bold', color=values['Color'])

# Formatting
plt.title('Comprehensive Performance Comparison: Before vs. After Balancing', fontsize=18, fontweight='bold', pad=20)
plt.ylabel('Score (%)', fontsize=13, fontweight='bold')
plt.xlabel('Models', fontsize=13, fontweight='bold')
plt.ylim(85, 99)

# Legend Organization
plt.legend(bbox_to_anchor=(0.5, -0.15), loc='upper center', borderaxespad=0.,
           ncol=4, fontsize=11, frameon=True, shadow=True, fancybox=True)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# Save and Show
plt.savefig('Thick_Dotted_Comparison_Graph.png', dpi=300, bbox_inches='tight')
plt.show()